# Spatial augmentation transforms

This notebook confirms that `SpatialAugmenter` applies horizontal flip, vertical flip,
90-degree rotation and additive noise consistently to both the input tensor and the
ground-truth maps, preserving spatial correspondence between them.

Modules exercised:

- `pipelines.dataset_pipeline.datasets.SpatialAugmenter`
- `configuration.dataset_config.AugmentationConfig`

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import matplotlib.pyplot as plt

import matplotlib
matplotlib.rcParams["figure.dpi"] = 110
matplotlib.rcParams["image.interpolation"] = "nearest"

SEED = 0
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    pass

print("repo root on path:", REPO_ROOT)



## Asymmetric test pattern

We use an L-shaped marker so that flips and rotations are unambiguous, and pair it with a
ground-truth map carrying the same marker, to verify both transform together.


In [ ]:
from tools.monitoring.logger                        import Logger
from configuration.dataset_config        import AugmentationConfig
from pipelines.dataset_pipeline.datasets import SpatialAugmenter

logger = Logger(log_dir="/tmp/dataset_nb_logs", name="nb09", level="WARNING")

H, W            = 32, 32
pattern         = np.zeros((1, H, W), dtype=np.float32)
pattern[0, 4:20, 4:8]  = 1.0
pattern[0, 16:20, 4:24] = 1.0
gt              = pattern.copy() * 0.5

def show_pair(inp, gt_map, title):
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    axes[0].imshow(inp[0], cmap="gray"); axes[0].set_title(title + " input")
    axes[1].imshow(gt_map[0], cmap="gray"); axes[1].set_title(title + " gt")
    for ax in axes: ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()

show_pair(pattern, gt, "original")



## Deterministic single transforms

By setting one probability to 1 and the rest to 0 (and seeding the augmenter RNG), we isolate
each transform. Noise is disabled here so the geometry is exact.


In [ ]:
def make_augmenter(**probs):
    cfg = AugmentationConfig(p_flip_h=0.0, p_flip_v=0.0, p_rot90=0.0, p_noise=0.0, noise_std=0.0)
    for k, v in probs.items():
        setattr(cfg, k, v)
    aug      = SpatialAugmenter(cfg, logger)
    aug._rng = np.random.default_rng(SEED)
    return aug

flip_h = make_augmenter(p_flip_h=1.0)
xi, yi = flip_h(pattern.copy(), gt.copy())
show_pair(xi, yi, "flip_h")
print("flip_h input matches mirror:", np.allclose(xi[0], pattern[0, :, ::-1]))
print("flip_h gt matches mirror   :", np.allclose(yi[0], gt[0, :, ::-1]))


In [ ]:
flip_v = make_augmenter(p_flip_v=1.0)
xi, yi = flip_v(pattern.copy(), gt.copy())
show_pair(xi, yi, "flip_v")
print("flip_v input matches mirror:", np.allclose(xi[0], pattern[0, ::-1, :]))


In [ ]:
rot = make_augmenter(p_rot90=1.0)
xi, yi = rot(pattern.copy(), gt.copy())
show_pair(xi, yi, "rot90")
print("input and gt rotated together:", np.allclose(xi[0] * 0.5, yi[0]))



## Additive noise

With noise enabled the input gains a small Gaussian perturbation while the ground truth stays
geometrically aligned (noise is applied to the input only).


In [ ]:
noisy      = make_augmenter(p_noise=1.0)
noisy.config.noise_std = 0.1
xi, yi     = noisy(pattern.copy(), gt.copy())

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(pattern[0], cmap="gray");        axes[0].set_title("input before")
axes[1].imshow(xi[0], cmap="gray");             axes[1].set_title("input after noise")
axes[2].imshow(xi[0] - pattern[0], cmap="coolwarm"); axes[2].set_title("difference (noise)")
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print("gt unchanged by noise:", np.allclose(yi[0], gt[0]))


## Expected visual outcome

Each isolated transform should move the L-shaped marker exactly as named, and the input and
ground-truth panels should always transform in lockstep (the printed boolean checks should be
True). The noise panel should show a faint random difference field over the input while the
ground truth remains identical.